# New DSAS workflow (v2 update)

This notebook builds an `NZCCDv2` **subset** from `NZCCDv1`, adds shoreline features for the AOIs picked up by the same new-data selection logic used in the new transect and uncertainty workflows, then runs DSAS-style calculations using the transects and `Total_UNCY` values produced by the two earlier notebooks.

Three things to know about the outputs:

- **They go in your own folder.** Set `RUN_OWNER` to your name and everything lands in `DataUpdatev2/<yourname>/`. Use the same value in all three notebooks. This is what stops two people who run the same AOI from overwriting each other.
- **Only the area you ran is kept.** NZCCDv1 rows outside the matched AOIs are dropped, so the file is your slice of the coast, not the whole country.
- **Filenames are tagged with your selection.** The tag comes from `search_mode`: `region`/`aoi` add the name, the `*_in_date_range` modes also add `since<YYYYMMDD>`.

Outputs (where `<tag>` is e.g. `Auckland_since20240718`):

- `DataUpdatev2/<yourname>/NZCCDv2_<tag>.shp` - shorelines for the AOIs in this run
- `DataUpdatev2/<yourname>/ratesv2_<tag>.shp` - transect-level DSAS rates + date/distance timeseries
- `DataUpdatev2/<yourname>/intersectsv2_<tag>.shp` - transect-shoreline intersection points + attributes
- `DataUpdatev2/<yourname>/new_dsas_exclusions_<tag>.csv` - shorelines left out of DSAS, and why

These per-area files are combined into the national dataset by `NZCCDv2_merge.ipynb`, which is run by the project maintainer.


In [ ]:
%load_ext autotime
import warnings
import re
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
import statsmodels.api as sm
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

SOURCE_DIR = Path("Data for testing")

# Put your own name here. All three notebooks must use the same value, so your outputs
# stay in your own folder and can never overwrite someone else's run of the same area.
RUN_OWNER = "yourname"
DATA_DIR = Path("DataUpdatev2") / RUN_OWNER
DATA_DIR.mkdir(parents=True, exist_ok=True)
V1_PATH = SOURCE_DIR / "NZCCDv1.shp"
TRANSECTS_PATH = DATA_DIR / "new_transects.shp"
UNCY_SUMMARY_PATH = DATA_DIR / "new_uncy_summary.csv"
# NZCCDv2 / rates / intersects filenames are built from the selection below, in the next cell

cutoff_date = pd.Timestamp("2024-07-18")
search_roots = [Path(r"Z:\\MaxarImagery\\HighFreq"), Path(r"Z:\\Retrolens")]
search_mode = "region_in_date_range"  # 'date', 'aoi', 'aoi_in_date_range', 'region', or 'region_in_date_range'
target_aoi = "MedlandsBeach"
target_region = "Auckland"

def _norm(text):
    return ''.join(ch for ch in str(text).lower() if ch.isalnum())

def normalize_path(value):
    return str(value).replace('\\', '/').lower()

def pick_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

def parse_date_from_stem(stem):
    text = str(stem)
    m = re.search(r"(\\d{1,2}[A-Za-z]{3,4}\\d{4})", text)
    if m:
        token = m.group(1).upper().replace("APRL", "APR").replace("SEPT", "SEP")
        for fmt in ("%d%b%Y", "%d%B%Y"):
            try:
                return pd.to_datetime(token, format=fmt)
            except Exception:
                pass
    m = re.search(r"(\\d{4}-\\d{2}-\\d{2})", text)
    if m:
        try:
            return pd.to_datetime(m.group(1), format="%Y-%m-%d")
        except Exception:
            pass
    m = re.search(r"(\\d{8})", text)
    if m:
        try:
            return pd.to_datetime(m.group(1), format="%Y%m%d")
        except Exception:
            pass
    return pd.NaT

def geom_hash(geom):
    if geom is None:
        return None
    try:
        return shapely.to_wkb(geom, hex=True)
    except Exception:
        return None

time: 16.2 s (started: 2026-07-30 18:28:30 +12:00)


In [ ]:
# 1) Name the outputs after the selection, then load NZCCDv1 as the starting point
# Tagging filenames keeps two people working on different areas from writing to the same file.
def _slug(text):
    return re.sub(r"[^A-Za-z0-9]+", "", str(text))

_date_tag = f"since{pd.Timestamp(cutoff_date):%Y%m%d}"
if search_mode == "date":
    OUTPUT_TAG = _date_tag
elif search_mode == "aoi":
    OUTPUT_TAG = _slug(target_aoi)
elif search_mode == "aoi_in_date_range":
    OUTPUT_TAG = f"{_slug(target_aoi)}_{_date_tag}"
elif search_mode == "region":
    OUTPUT_TAG = _slug(target_region)
elif search_mode == "region_in_date_range":
    OUTPUT_TAG = f"{_slug(target_region)}_{_date_tag}"
else:
    raise ValueError(f"Unknown search_mode: {search_mode}")

V2_PATH = DATA_DIR / f"NZCCDv2_{OUTPUT_TAG}.shp"
RATES_OUT = DATA_DIR / f"ratesv2_{OUTPUT_TAG}.shp"
POINTS_OUT = DATA_DIR / f"intersectsv2_{OUTPUT_TAG}.shp"
EXCLUSIONS_OUT = DATA_DIR / f"new_dsas_exclusions_{OUTPUT_TAG}.csv"

if not V1_PATH.exists():
    raise FileNotFoundError(f"Missing source dataset: {V1_PATH}")

v2 = gpd.read_file(V1_PATH)

print(f"Output tag: {OUTPUT_TAG}")
print(f"Will write: {V2_PATH.name} / {RATES_OUT.name} / {POINTS_OUT.name}")
print(f"Loaded NZCCDv1 rows: {len(v2):,}")
v2.head(2)


Created NZCCDv2 from NZCCDv1 with 9 sidecar files
Loaded NZCCDv2 rows: 19,666


,Region,Site,Digitiser,Scale,Notes,Source,CPS,Proxy,Photoscale,Georef_ER,Pixel_Er,Total_UNCY,USDate,SHLength,Date,ID,geometry
0,Auckland,KarekareBethells,MW,2000,tod,RL,4,1,40000,5.03,1.416425,5.620681,01/02/2004,1.265403,2004-01-02,0.0,"LINESTRING Z (1728907.5 5916213.248 0, 1728868.341 5916181.498 0, 1728823.891 5916158.215 0, 1728765.154 5916132.815 0, 1728735.52 59161..."
1,Auckland,KarekareBethells,MW,2000,tod,RL,4,1,40000,5.03,1.416425,5.620681,01/02/2004,0.307010,2004-01-02,1.0,"LINESTRING Z (1729067.838 5914779.733 0, 1729077.892 5914779.733 0, 1729089.534 5914776.028 0, 1729092.179 5914765.445 0, 1729082.654 59..."


time: 1.16 s (started: 2026-07-30 18:28:46 +12:00)


In [3]:
# 2) Find new shoreline files using the same mode logic as new_transects/new_uncy
valid_modes = {'date', 'aoi', 'aoi_in_date_range', 'region', 'region_in_date_range'}
if search_mode not in valid_modes:
    raise ValueError(f"search_mode must be one of {sorted(valid_modes)}")
if search_mode in {'aoi', 'aoi_in_date_range'} and not str(target_aoi).strip():
    raise ValueError('target_aoi must be set when using AOI-based modes')
if search_mode in {'region', 'region_in_date_range'} and not str(target_region).strip():
    raise ValueError('target_region must be set when using region-based modes')

target_aoi_norm = _norm(target_aoi)
target_region_norm = _norm(target_region)
records = []

for root in search_roots:
    if not root.exists():
        continue
    for shp in root.glob('**/Shorelines/*.shp'):
        if shp.stem.lower().startswith('[aoierr]'):
            continue
        if len(shp.parts) < 5 or shp.parts[-2].lower() != 'shorelines':
            continue

        region = shp.parts[-4]
        aoi = shp.parts[-3]
        modified = pd.Timestamp(shp.stat().st_mtime, unit='s')
        stem_aoi = shp.stem.rsplit('_', 1)[0]

        matches_aoi = target_aoi_norm in {_norm(aoi), _norm(stem_aoi)}
        matches_region = target_region_norm == _norm(region)
        matches_date = modified > cutoff_date

        include = False
        if search_mode == 'date':
            include = matches_date
        elif search_mode == 'aoi':
            include = matches_aoi
        elif search_mode == 'aoi_in_date_range':
            include = matches_aoi and matches_date
        elif search_mode == 'region':
            include = matches_region
        elif search_mode == 'region_in_date_range':
            include = matches_region and matches_date

        if include:
            records.append({
                'region': region,
                'aoi': aoi,
                'shoreline_path': str(shp),
                'modified': modified,
            })

new_shorelines = pd.DataFrame(records)
if new_shorelines.empty:
    raise ValueError('No new shoreline files matched current mode settings.')

new_shorelines = new_shorelines.sort_values(['region', 'aoi', 'modified']).reset_index(drop=True)
target_aois = new_shorelines[['region', 'aoi']].drop_duplicates().reset_index(drop=True)
print(f"Matched new shoreline files: {len(new_shorelines)} across {len(target_aois)} AOIs")
display(target_aois.head(20))

Matched new shoreline files: 14 across 4 AOIs


,region,aoi
0,Auckland,KarekareBethells
1,Auckland,ManukapuaIsland
2,Auckland,MuriwaiSouth
3,Auckland,Omaha


time: 2min 29s (started: 2026-07-30 18:28:47 +12:00)


In [ ]:
# 2b) Trim NZCCDv1 down to the areas this run actually covers
# Rows outside the matched AOIs are dropped, so the NZCCDv2 written below holds only
# the coastline this run is responsible for and can be merged with other people's runs later.
region_col = pick_col(v2, ["Region"])
location_col = pick_col(v2, ["Location", "Site"])
if region_col is None or location_col is None:
    raise ValueError("NZCCDv1 has no Region/Location(Site) columns, so it cannot be trimmed to the target AOIs")

target_pairs = {(_norm(r.region), _norm(r.aoi)) for r in target_aois.itertuples(index=False)}
v2_pairs = list(zip(v2[region_col].map(_norm), v2[location_col].map(_norm)))
keep_mask = pd.Series([pair in target_pairs for pair in v2_pairs], index=v2.index)

kept_pairs = {pair for pair, keep in zip(v2_pairs, keep_mask) if keep}
missing_pairs = sorted(target_pairs - kept_pairs)

dropped = int((~keep_mask).sum())
v2 = v2[keep_mask].reset_index(drop=True)

# NZCCDv1 calls the AOI column "Site"; the merge/dedupe steps below expect "Location"
if "Location" not in v2.columns:
    v2["Location"] = v2[location_col]

print(f"Kept {len(v2):,} NZCCDv1 rows across {len(kept_pairs)} AOI(s); dropped {dropped:,} rows outside this run")
if missing_pairs:
    print(f"No NZCCDv1 history for {len(missing_pairs)} target AOI(s) - those shorelines come from the drive only:")
    for region, aoi in missing_pairs:
        print(f"  - {region} / {aoi}")
if v2.empty:
    print("WARNING: no NZCCDv1 rows matched. Check that Region/Site spelling in NZCCDv1 matches the drive folder names.")


In [ ]:
# 3) Build full AOI shoreline set (all dates) and append missing rows into NZCCDv2
if not UNCY_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Missing uncertainty summary: {UNCY_SUMMARY_PATH}")

uncy_summary = pd.read_csv(UNCY_SUMMARY_PATH)
required_uncy_cols = {'path', 'total_uncy_mean'}
missing_uncy_cols = required_uncy_cols - set(uncy_summary.columns)
if missing_uncy_cols:
    raise ValueError(f"{UNCY_SUMMARY_PATH} is missing columns: {sorted(missing_uncy_cols)}")

uncy_summary = uncy_summary.copy()
uncy_summary['path_norm'] = uncy_summary['path'].map(normalize_path)
uncy_summary['total_uncy_mean'] = pd.to_numeric(uncy_summary['total_uncy_mean'], errors='coerce')
uncy_map = (
    uncy_summary.dropna(subset=['path_norm', 'total_uncy_mean'])
    .drop_duplicates('path_norm', keep='last')
    .set_index('path_norm')['total_uncy_mean']
    .to_dict()
)

new_file_paths_norm = set(new_shorelines['shoreline_path'].map(normalize_path).tolist())
all_aoi_rows = []
exclusion_rows = []

for row in target_aois.itertuples(index=False):
    region = row.region
    aoi = row.aoi

    shoreline_files = []
    for root in search_roots:
        shoreline_dir = root / region / aoi / 'Shorelines'
        if shoreline_dir.exists():
            shoreline_files.extend(sorted(shoreline_dir.glob('*.shp')))

    for shp in shoreline_files:
        if shp.stem.lower().startswith('[aoierr]'):
            continue
        g = gpd.read_file(shp)
        if g.empty:
            continue

        if g.crs is not None and v2.crs is not None and str(g.crs) != str(v2.crs):
            g = g.to_crs(v2.crs)

        date_col = pick_col(g, ['Date'])
        if date_col is None:
            g['Date'] = parse_date_from_stem(shp.stem)
        else:
            g['Date'] = pd.to_datetime(g[date_col], errors='coerce')

        shp_norm = normalize_path(shp)
        g['Total_UNCY'] = uncy_map.get(shp_norm, np.nan)

        g['Region'] = region
        g['Location'] = aoi
        g['SourceFile'] = str(shp)
        g['GeomHash'] = g.geometry.apply(geom_hash)

        uncy_series = pd.to_numeric(g['Total_UNCY'], errors='coerce')
        eligible_mask = g['Date'].notna() & uncy_series.notna() & (uncy_series > 0)
        excluded_rows = int((~eligible_mask).sum())
        if excluded_rows > 0:
            reasons = []
            missing_date_rows = int(g['Date'].isna().sum())
            missing_uncy_rows = int(uncy_series.isna().sum())
            non_positive_uncy_rows = int((uncy_series.notna() & (uncy_series <= 0)).sum())
            if missing_uncy_rows > 0:
                reasons.append('missing Total_UNCY')
            if non_positive_uncy_rows > 0:
                reasons.append('non-positive Total_UNCY')
            if missing_date_rows > 0:
                reasons.append('missing Date')
            exclusion_rows.append({
                'filename': str(shp),
                'region': region,
                'aoi': aoi,
                'is_new_shoreline': shp_norm in new_file_paths_norm,
                'total_rows': int(len(g)),
                'excluded_rows': excluded_rows,
                'included_rows': int(eligible_mask.sum()),
                'reason': '; '.join(reasons),
            })

        keep_cols = [c for c in v2.columns if c in g.columns]
        if 'Date' not in keep_cols:
            keep_cols.append('Date')
        if 'Total_UNCY' not in keep_cols:
            keep_cols.append('Total_UNCY')
        if 'Region' not in keep_cols:
            keep_cols.append('Region')
        if 'Location' not in keep_cols:
            keep_cols.append('Location')
        keep_cols.extend([c for c in ['SourceFile', 'GeomHash', 'geometry'] if c not in keep_cols and c in g.columns])

        all_aoi_rows.append(g[keep_cols].copy())

if len(all_aoi_rows) == 0:
    raise ValueError('No shoreline features found for target AOIs.')

incoming = gpd.GeoDataFrame(pd.concat(all_aoi_rows, ignore_index=True), crs=v2.crs)
if 'GeomHash' not in v2.columns:
    v2['GeomHash'] = v2.geometry.apply(geom_hash)
if 'Region' not in v2.columns:
    v2['Region'] = pd.NA
if 'Location' not in v2.columns:
    v2['Location'] = pd.NA
if 'Date' in v2.columns:
    v2['Date'] = pd.to_datetime(v2['Date'], errors='coerce')
else:
    v2['Date'] = pd.NaT
if 'Total_UNCY' in v2.columns:
    v2['Total_UNCY'] = pd.to_numeric(v2['Total_UNCY'], errors='coerce')
else:
    v2['Total_UNCY'] = np.nan

for col in incoming.columns:
    if col not in v2.columns:
        v2[col] = pd.NA
for col in v2.columns:
    if col not in incoming.columns:
        incoming[col] = pd.NA
incoming = incoming[v2.columns]

v2_key = pd.DataFrame({
    'Region': v2['Region'].astype(str).map(_norm),
    'Location': v2['Location'].astype(str).map(_norm),
    'Date': pd.to_datetime(v2['Date'], errors='coerce').astype(str),
    'GeomHash': v2['GeomHash'].astype(str),
})
incoming_key = pd.DataFrame({
    'Region': incoming['Region'].astype(str).map(_norm),
    'Location': incoming['Location'].astype(str).map(_norm),
    'Date': pd.to_datetime(incoming['Date'], errors='coerce').astype(str),
    'GeomHash': incoming['GeomHash'].astype(str),
})

existing = set(map(tuple, v2_key[['Region', 'Location', 'Date', 'GeomHash']].itertuples(index=False, name=None)))
is_new = incoming_key[['Region', 'Location', 'Date', 'GeomHash']].apply(tuple, axis=1).map(lambda t: t not in existing)
to_add = incoming[is_new].copy()

v2_updated = gpd.GeoDataFrame(pd.concat([v2, to_add], ignore_index=True), crs=v2.crs)
v2_updated.to_file(V2_PATH)

if len(exclusion_rows) == 0:
    dsas_exclusions = pd.DataFrame(columns=['filename', 'region', 'aoi', 'is_new_shoreline', 'total_rows', 'excluded_rows', 'included_rows', 'reason'])
else:
    dsas_exclusions = pd.DataFrame(exclusion_rows).drop_duplicates().reset_index(drop=True)

print(f"Loaded uncertainty values for {len(uncy_map):,} shoreline files from {UNCY_SUMMARY_PATH}")
print(f"Added {len(to_add):,} shoreline rows to NZCCDv2")
print(f"NZCCDv2 total rows: {len(v2_updated):,}")
print(f"Shoreline files with pre-DSAS exclusions: {dsas_exclusions['filename'].nunique() if len(dsas_exclusions) > 0 else 0}")
display(dsas_exclusions.head(20))
v2_updated[['Region', 'Location', 'Date', 'Total_UNCY']].tail(10)

Loaded uncertainty values for 12 shoreline files from new_uncy_summary.csv
Added 580 shoreline rows to NZCCDv2
NZCCDv2 total rows: 20,246
Shoreline files with pre-DSAS exclusions: 44


c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field Date created as String field, though DateTime requested.
  ogr_write(
c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value '010200008031000000C0E0F97F8B613A4182C5DF4F8D915641000000000000000050366C5764613A4128BBDE5F859156410000000000000000942F33E437613A41B6D5BB8D7F9156410000000000000000E0AF5E27FD603A410B67213479915641000000000000000068AB3885DF603A41B62F540776915641000000000000000098C9DD2FB5603A41835564046F9156410000000000000000083665B8AC603A411E47EC846A915641000000000000000028B7EE20B4603A4116F2FC79679156410000000000000000F400AB5CB8603A41F0EEB7C9619156410000000000000000D8CBF000C4603A410DF7A5E85A9156410000000000000000803A8B5ACA603A4144ADE9AC569156410000000000000000D8F8BC3DBF603A41F6F7D7C751915641000000000000000060B901F2C2603A4149893D6E4B9156410000000000000000D8CBF000C4603A412D08B405449156410000000000000000D8CBF000C4603A41FFAF7F4A3B9156410000000000000000F07946A6C6603A41F105A134359156410000000

,filename,region,aoi,is_new_shoreline,total_rows,excluded_rows,included_rows,reason
0,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_01MAR2015.shp,Auckland,KarekareBethells,False,15,15,0,missing Total_UNCY
1,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_03JAN2011.shp,Auckland,KarekareBethells,False,8,8,0,missing Total_UNCY
2,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_04JAN2017.shp,Auckland,KarekareBethells,False,19,19,0,missing Total_UNCY
3,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_04JAN2022.shp,Auckland,KarekareBethells,False,19,19,0,missing Total_UNCY
4,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_04JUL2024.shp,Auckland,KarekareBethells,True,6,6,0,missing Total_UNCY
5,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_05APR2016.shp,Auckland,KarekareBethells,False,6,6,0,missing Total_UNCY
6,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_07APRIL2010.shp,Auckland,KarekareBethells,False,9,9,0,missing Total_UNCY
7,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_19MAR2023.shp,Auckland,KarekareBethells,True,6,6,0,missing Total_UNCY
8,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_20SEP2008.shp,Auckland,KarekareBethells,False,9,9,0,missing Total_UNCY
9,Z:\MaxarImagery\HighFreq\Auckland\KarekareBethells\Shorelines\KarekareBethells_21MAR2021.shp,Auckland,KarekareBethells,False,23,23,0,missing Total_UNCY


,Region,Location,Date,Total_UNCY
20236,Auckland,Omaha,2021-12-23,NaN
20237,Auckland,Omaha,2021-12-23,NaN
20238,Auckland,Omaha,2021-12-23,NaN
20239,Auckland,Omaha,2021-12-23,NaN
20240,Auckland,Omaha,2021-12-23,NaN
20241,Auckland,Omaha,2023-02-28,NaN
20242,Auckland,Omaha,2023-02-28,NaN
20243,Auckland,Omaha,2023-02-28,NaN
20244,Auckland,Omaha,2023-02-28,NaN
20245,Auckland,Omaha,2023-02-28,NaN


time: 6.45 s (started: 2026-07-30 18:34:26 +12:00)


In [12]:
# 4) DSAS calculations using new transects and updated NZCCDv2 shorelines
if not TRANSECTS_PATH.exists():
    raise FileNotFoundError(f"Missing transects: {TRANSECTS_PATH}")

transects = gpd.read_file(TRANSECTS_PATH)
uid_col = pick_col(transects, ['Unique_ID', 'UniqueID'])
if uid_col is None:
    raise ValueError('Transects must include Unique_ID or UniqueID')
transects = transects.rename(columns={uid_col: 'Unique_ID'}).set_index('Unique_ID')
if transects.crs is None:
    transects = transects.set_crs(2193, allow_override=True)
else:
    transects = transects.to_crs(2193)

# Ensure one transect geometry per Unique_ID.
if transects.index.duplicated().any():
    dup_count = int(transects.index.duplicated().sum())
    print(f"Dropping {dup_count:,} duplicate transect rows by Unique_ID (keeping first geometry)")
    transects = transects[~transects.index.duplicated(keep='first')].copy()

shore = gpd.read_file(V2_PATH)
if shore.crs is None:
    shore = shore.set_crs(2193, allow_override=True)
else:
    shore = shore.to_crs(2193)

region_col = pick_col(shore, ['Region', 'region'])
aoi_col = pick_col(shore, ['Location', 'AOI', 'aoi', 'location'])
date_col = pick_col(shore, ['Date', 'date'])
uncy_col = pick_col(shore, ['Total_UNCY', 'total_uncy', 'new_Total_UNCY'])
sourcefile_col = pick_col(shore, ['SourceFile', 'sourcefile'])

if region_col is None or aoi_col is None or date_col is None:
    raise ValueError('NZCCDv2 must include Region/Location(Date) columns for DSAS')

shore = shore.rename(columns={region_col: 'Region', aoi_col: 'AOI', date_col: 'Date'})
shore = shore.loc[:, ~shore.columns.duplicated()].copy()
shore = shore.set_geometry('geometry')
shore['Date'] = pd.to_datetime(shore['Date'], errors='coerce')
if uncy_col is None:
    shore['Total_UNCY'] = np.nan
else:
    shore['Total_UNCY'] = pd.to_numeric(shore[uncy_col], errors='coerce')
if sourcefile_col is None:
    shore['SourceFile'] = pd.NA
else:
    shore['SourceFile'] = shore[sourcefile_col].astype(str)

target_key = set(target_aois.apply(lambda r: (_norm(r.region), _norm(r.aoi)), axis=1).tolist())
shore = shore[shore.apply(lambda r: (_norm(r['Region']), _norm(r['AOI'])) in target_key, axis=1)].copy()
shore = shore[shore.geometry.notna()].copy()
shore['dsas_eligible'] = shore['Date'].notna() & shore['Total_UNCY'].notna() & (shore['Total_UNCY'] > 0)

def to_point_or_empty(geom, transect_origin):
    if geom is None or geom.is_empty:
        return shapely.Point()
    gt = geom.geom_type
    if gt == 'Point':
        return geom
    if gt == 'MultiPoint':
        pts = list(geom.geoms)
        if len(pts) == 0:
            return shapely.Point()
        pts = sorted(pts, key=lambda p: p.distance(transect_origin))
        return pts[0]
    try:
        p = shapely.get_point(geom, 0)
        return p if p is not None else shapely.Point()
    except Exception:
        return shapely.Point()

def intersect_or_empty(geom, line):
    if geom is None:
        return shapely.GeometryCollection()
    try:
        return geom.intersection(line)
    except Exception:
        return shapely.GeometryCollection()

def process_transect(unique_id):
    transect = transects.geometry.loc[unique_id]
    tran_origin = shapely.get_point(transect, -1)

    local = shore.copy()
    intersections = [intersect_or_empty(geom, transect) for geom in local.geometry.values]
    local['intersect_raw'] = intersections
    local['intersect_point'] = [to_point_or_empty(g, tran_origin) for g in intersections]

    # Promote intersection points to a GeoSeries so geometric vector ops are available.
    point_gs = gpd.GeoSeries(local['intersect_point'], index=local.index, crs=shore.crs)
    local = local[~point_gs.is_empty].copy()
    point_gs = point_gs.loc[local.index]
    local = local[local['dsas_eligible']].sort_values('Date')
    point_gs = point_gs.loc[local.index]

    if len(local) < 3:
        return None, None

    local['YearsSinceBase'] = (local['Date'] - local['Date'].min()).dt.days / 365.25
    local['Distance'] = point_gs.distance(tran_origin)

    lr = sm.OLS(local['Distance'], sm.add_constant(local['YearsSinceBase'])).fit()
    lr_low, lr_high = lr.conf_int(alpha=0.1).loc['YearsSinceBase']
    lci = (lr_high - lr_low) / 2

    wlr = sm.WLS(local['Distance'], sm.add_constant(local['YearsSinceBase']), weights=1 / (local['Total_UNCY'] ** 2)).fit()
    wlr_low, wlr_high = wlr.conf_int(alpha=0.1).loc['YearsSinceBase']
    wci = (wlr_high - wlr_low) / 2

    duration = (local['Date'].max() - local['Date'].min()).days / 365.25
    if duration <= 0:
        return None, None

    nsm = -(local['Distance'].iloc[0] - local['Distance'].iloc[-1])
    sce = point_gs.apply(lambda p: point_gs.distance(p).max()).max()

    rate_row = {
        'UniqueID': unique_id,
        'Region': local['Region'].astype(str).value_counts().idxmax(),
        'AOI': local['AOI'].astype(str).value_counts().idxmax(),
        'Start_date': str(local['Date'].min().date()),
        'End_date': str(local['Date'].max().date()),
        'Duration': round(duration),
        'ShrCount': len(local),
        'NSM': round(nsm, 2),
        'SCE': round(sce, 2),
        'EPR': round(nsm / duration, 2),
        'EPRunc': round(np.sqrt(local['Total_UNCY'].iloc[0] ** 2 + local['Total_UNCY'].iloc[-1] ** 2) / duration, 2),
        'LRR': round(lr.params['YearsSinceBase'], 2),
        'LRI': round(lr.params['const'], 2),
        'LCI': round(lci, 2),
        'LSE': round(np.sqrt(lr.mse_resid), 2),
        'LR2': round(lr.rsquared, 2),
        'WLR': round(wlr.params['YearsSinceBase'], 2),
        'WLI': round(wlr.params['const'], 2),
        'WCI': round(wci, 2),
        'WSE': round(np.sqrt(wlr.mse_resid), 2),
        'WR2': round(wlr.rsquared, 2),
        'Dates': local['Date'].dt.strftime('%Y-%m-%d').tolist(),
        'Distances': local['Distance'].round(2).tolist(),
        'geometry': transect,
    }

    point_rows = local[['Region', 'AOI', 'Date', 'Distance', 'Total_UNCY', 'SourceFile', 'intersect_point']].copy()
    point_rows['Unique_ID'] = unique_id
    point_rows['YearsSinceBase'] = local['YearsSinceBase']
    point_rows['NSM'] = round(nsm, 2)
    point_rows['EPR'] = round(nsm / duration, 2)
    point_rows['LRR'] = round(lr.params['YearsSinceBase'], 2)
    point_rows['WLR'] = round(wlr.params['YearsSinceBase'], 2)
    point_rows = point_rows.rename(columns={'intersect_point': 'geometry'})

    return rate_row, point_rows

rate_rows = []
point_frames = []
for uid in tqdm(transects.index.tolist()):
    rate_row, point_rows = process_transect(uid)
    if rate_row is None:
        continue
    rate_rows.append(rate_row)
    point_frames.append(point_rows)

if len(rate_rows) == 0:
    raise ValueError('No transects produced DSAS statistics (need at least 3 eligible shoreline intersections per transect).')

rates = gpd.GeoDataFrame(rate_rows, crs=transects.crs)
points = gpd.GeoDataFrame(pd.concat(point_frames, ignore_index=True), crs=transects.crs)
points['Date'] = pd.to_datetime(points['Date'], errors='coerce')

# Add post-DSAS exclusions for eligible shoreline files that never made it into DSAS outputs.
if 'dsas_exclusions' not in globals():
    dsas_exclusions = pd.DataFrame(columns=['filename', 'region', 'aoi', 'is_new_shoreline', 'total_rows', 'excluded_rows', 'included_rows', 'reason'])

eligible_files = set(
    shore.loc[shore['dsas_eligible'] & shore['SourceFile'].notna() & (shore['SourceFile'].astype(str) != 'nan'), 'SourceFile']
    .astype(str)
    .unique()
    .tolist()
)
used_files = set(points['SourceFile'].dropna().astype(str).unique().tolist())
not_used_files = sorted(eligible_files - used_files)

if len(not_used_files) > 0:
    extra_exclusions = pd.DataFrame({
        'filename': not_used_files,
        'region': [pd.NA] * len(not_used_files),
        'aoi': [pd.NA] * len(not_used_files),
        'is_new_shoreline': [pd.NA] * len(not_used_files),
        'total_rows': [pd.NA] * len(not_used_files),
        'excluded_rows': [pd.NA] * len(not_used_files),
        'included_rows': [pd.NA] * len(not_used_files),
        'reason': ['eligible shoreline not used in final DSAS results (no successful transect intersections)'] * len(not_used_files),
    })
    dsas_exclusions = pd.concat([dsas_exclusions, extra_exclusions], ignore_index=True).drop_duplicates()

display(rates.head(10))
display(points.head(10))
print(f"Rates rows: {len(rates):,}")
print(f"Point rows: {len(points):,}")
print(f"DSAS exclusions rows: {len(dsas_exclusions):,}")

Dropping 3,896 duplicate transect rows by Unique_ID (keeping first geometry)


  0%|          | 0/32480 [00:00<?, ?it/s]

,UniqueID,Region,AOI,Start_date,End_date,Duration,ShrCount,NSM,SCE,EPR,EPRunc,LRR,LRI,LCI,LSE,LR2,WLR,WLI,WCI,WSE,WR2,Dates,Distances,geometry
0,100185142000,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-3.90,3.90,-0.21,0.15,-0.21,170.98,0.08,0.54,0.89,-0.22,171.04,0.07,0.22,0.93,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[170.78, 170.46, 168.66, 168.68, 168.39, 166.89]","LINESTRING (1760790.821 5975849.733, 1760927.548 5975473.826)"
1,100185140855,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-4.03,4.03,-0.22,0.15,-0.18,169.19,0.09,0.60,0.82,-0.20,169.35,0.08,0.28,0.87,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[169.39, 167.56, 167.42, 167.77, 167.31, 165.37]","LINESTRING (1760781.423 5975846.315, 1760918.15 5975470.409)"
2,100185139785,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-3.95,3.95,-0.21,0.15,-0.15,168.63,0.14,0.93,0.59,-0.18,168.80,0.12,0.41,0.72,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[169.13, 166.81, 166.81, 167.39, 167.67, 165.18]","LINESTRING (1760772.04 5975842.903, 1760908.737 5975466.986)"
3,100185138679,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-4.96,4.96,-0.27,0.15,-0.19,169.08,0.18,1.25,0.54,-0.22,169.29,0.16,0.54,0.69,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[169.84, 166.54, 166.96, 167.54, 167.99, 164.88]","LINESTRING (1760762.627 5975839.481, 1760899.354 5975463.574)"
4,100185137650,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-5.89,5.89,-0.32,0.15,-0.24,170.13,0.20,1.34,0.62,-0.25,170.22,0.16,0.56,0.73,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[171.27, 166.93, 167.65, 167.73, 168.26, 165.39]","LINESTRING (1760753.244 5975836.069, 1760889.941 5975460.151)"
5,100185136634,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-6.18,6.18,-0.34,0.15,-0.26,170.86,0.21,1.39,0.64,-0.26,170.89,0.17,0.57,0.74,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[172.15, 167.23, 168.57, 168.05, 168.49, 165.97]","LINESTRING (1760743.831 5975832.646, 1760880.558 5975456.739)"
6,100185135630,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-6.18,6.18,-0.34,0.15,-0.27,171.59,0.21,1.40,0.66,-0.26,171.50,0.17,0.58,0.74,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[172.98, 167.7, 169.56, 168.37, 168.5, 166.8]","LINESTRING (1760734.433 5975829.228, 1760871.161 5975453.322)"
7,100185134626,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-5.80,5.80,-0.32,0.15,-0.27,171.97,0.23,1.52,0.61,-0.25,171.89,0.18,0.63,0.69,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[173.16, 167.8, 170.8, 169.1, 168.37, 167.36]","LINESTRING (1760725.05 5975825.816, 1760861.748 5975449.899)"
8,100185133669,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-5.75,5.75,-0.31,0.15,-0.26,172.55,0.24,1.65,0.56,-0.27,172.67,0.20,0.69,0.68,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[173.22, 168.3, 172.04, 170.41, 169.07, 167.47]","LINESTRING (1760715.638 5975822.393, 1760852.365 5975446.487)"
9,100185132738,Auckland,Omaha,2006-02-03,2024-06-26,18,6,-5.35,5.45,-0.29,0.15,-0.24,173.94,0.29,1.96,0.45,-0.24,173.96,0.23,0.81,0.55,"[2006-02-03, 2011-12-20, 2014-01-18, 2017-01-20, 2020-02-02, 2024-06-26]","[174.79, 169.33, 174.06, 171.72, 170.31, 169.44]","LINESTRING (1760706.255 5975818.981, 1760842.952 5975443.064)"


,Region,AOI,Date,Distance,Total_UNCY,SourceFile,geometry,Unique_ID,YearsSinceBase,NSM,EPR,LRR,WLR
0,Auckland,Omaha,2006-02-03,170.783558,2.452305,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_03FEB2006.shp,POINT Z (1760869.171 5975634.323 0),100185142000,0.000000,-3.90,-0.21,-0.21,-0.22
1,Auckland,Omaha,2011-12-20,170.456203,2.429548,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_20DEC2011.shp,POINT Z (1760869.283 5975634.015 0),100185142000,5.875428,-3.90,-0.21,-0.21,-0.22
2,Auckland,Omaha,2014-01-18,168.656404,2.429774,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_18JAN2014.shp,POINT Z (1760869.898 5975632.324 0),100185142000,7.956194,-3.90,-0.21,-0.21,-0.22
3,Auckland,Omaha,2017-01-20,168.676710,2.007821,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_20JAN2017.shp,POINT Z (1760869.891 5975632.343 0),100185142000,10.962355,-3.90,-0.21,-0.21,-0.22
4,Auckland,Omaha,2020-02-02,168.392558,2.429774,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_02FEB2020.shp,POINT Z (1760869.989 5975632.076 0),100185142000,13.995893,-3.90,-0.21,-0.21,-0.22
5,Auckland,Omaha,2024-06-26,166.886635,1.427192,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_26JUN2024.shp,POINT Z (1760870.503 5975630.661 0),100185142000,18.392882,-3.90,-0.21,-0.21,-0.22
6,Auckland,Omaha,2006-02-03,169.393206,2.452305,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_03FEB2006.shp,POINT Z (1760860.249 5975629.599 0),100185140855,0.000000,-4.03,-0.22,-0.18,-0.20
7,Auckland,Omaha,2011-12-20,167.557455,2.429548,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_20DEC2011.shp,POINT Z (1760860.876 5975627.874 0),100185140855,5.875428,-4.03,-0.22,-0.18,-0.20
8,Auckland,Omaha,2014-01-18,167.418397,2.429774,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_18JAN2014.shp,POINT Z (1760860.924 5975627.743 0),100185140855,7.956194,-4.03,-0.22,-0.18,-0.20
9,Auckland,Omaha,2017-01-20,167.765973,2.007821,Z:\MaxarImagery\HighFreq\Auckland\Omaha\Shorelines\Omaha_20JAN2017.shp,POINT Z (1760860.805 5975628.07 0),100185140855,10.962355,-4.03,-0.22,-0.18,-0.20


Rates rows: 394
Point rows: 2,329
DSAS exclusions rows: 50
time: 3min 36s (started: 2026-07-30 18:44:27 +12:00)


In [ ]:
# 5) Save DSAS outputs
rates.to_file(RATES_OUT)
points.to_file(POINTS_OUT)

# Save exclusions report (shorelines not included in DSAS and why).
if 'dsas_exclusions' in globals():
    dsas_exclusions.to_csv(EXCLUSIONS_OUT, index=False)
else:
    pd.DataFrame(columns=['filename', 'region', 'aoi', 'is_new_shoreline', 'total_rows', 'excluded_rows', 'included_rows', 'reason']).to_csv(EXCLUSIONS_OUT, index=False)

# Optional tabular exports for easy QA
rates_csv = RATES_OUT.with_suffix('.csv')
points_csv = POINTS_OUT.with_suffix('.csv')
rates.drop(columns='geometry').to_csv(rates_csv, index=False)
points.drop(columns='geometry').to_csv(points_csv, index=False)

print(f"Saved shorelines shapefile: {V2_PATH}")
print(f"Saved rates shapefile: {RATES_OUT}")
print(f"Saved points shapefile: {POINTS_OUT}")
print(f"Saved exclusions report: {EXCLUSIONS_OUT}")
print(f"Saved CSV companions: {rates_csv.name}, {points_csv.name}")
